# Train BERTimbau Large v6 — SPR 2026

**Objetivo:** Treinar modelo diverso do v5 para ensemble 10-fold (v5+v6).

## 🔑 Diferenças em relação ao v5

| Parâmetro | v5 (base) | v6 (este) | Motivo |
|-----------|-----------|-----------|--------|
| SEED | 42 | 123 | Diversidade para ensemble |
| Loss | FocalLoss | FocalLoss + LabelSmoothing(0.1) | Regularização extra |
| Scheduler | Linear warmup | Cosine Annealing | Melhor convergência |
| Grad Accum | 1 | 2 (batch efetivo=16) | Gradientes mais estáveis |

**Uso previsto:** Ensemble v5+v6 = 10 folds com diversidade real entre modelos.

## ⚙️ Configuração no Kaggle

| Setting | Valor |
|---------|-------|
| **Internet** | OFF |
| **Accelerator** | GPU T4 x2 (ou P100) |
| **Add Input** | Competition: `spr-2026-mammography-report-classification` |
| **Add Input** | Models → `bertimbau-ptbr-complete` (fabianofilho) |

## 📂 Output gerado

```
/kaggle/working/advanced_outputs_kaggle_6/
    ├── all_results.pkl
    ├── solo_bert_artifacts.pkl
    └── weights/bertimbau_large__cb_focal/
        ├── fold_0.pt ... fold_4.pt
        ├── tokenizer/
        └── model_config/
```

In [ ]:
import os
import re
import gc
import math
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG — v6: diversidade via seed + label smoothing + cosine + grad accum
# ═══════════════════════════════════════════════════════════════════════════════
SEED             = 123       # diferente do v5 (42) → diversidade para ensemble
MAX_LEN          = 512
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 2         # batch efetivo = 8 × 2 = 16
EPOCHS           = 5
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
LABEL_SMOOTHING  = 0.1       # regularização extra
N_FOLDS          = 5
NUM_CLASSES      = 7
FOCAL_GAMMA      = 2.0
FOCAL_ALPHA      = 0.25
VERSION          = 6
CONFIG_KEY       = 'bertimbau_large__cb_focal'

# Paths
DATA_DIR    = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification')
OUTPUT_BASE = Path(f'/kaggle/working/advanced_outputs_kaggle_{VERSION}')
WEIGHTS_DIR = OUTPUT_BASE / 'weights' / CONFIG_KEY
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CANDIDATES = [
    '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
    '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
    '/kaggle/input/bert-large-portuguese-cased',
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if Path(p).exists()), None)
assert MODEL_PATH, (
    'BERTimbau Large não encontrado!\n'
    'Adicione: Add Input → Models → bertimbau-ptbr-complete (fabianofilho)'
)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = torch.cuda.is_available()

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device:          {DEVICE}')
print(f'FP16:            {USE_FP16}')
print(f'Model path:      {MODEL_PATH}')
print(f'Output:          {OUTPUT_BASE}')
print(f'Seed:            {SEED} (diferente do v5=42)')
print(f'Label Smoothing: {LABEL_SMOOTHING}')
print(f'Grad Accum:      {GRAD_ACCUM_STEPS} (batch efetivo={BATCH_SIZE*GRAD_ACCUM_STEPS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS
# ═══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df)} amostras')
print(f'Test:  {len(test_df)} amostras')
print(f'\nDistribuição das classes:')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_texts  = [build_input_text(t) for t in train_df['report'].tolist()]
train_labels = train_df['target'].tolist()
test_texts   = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Textos processados: {len(train_texts)} train, {len(test_texts)} test')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS + LABEL SMOOTHING
# Combina Focal Loss (foco em exemplos difíceis) com Label Smoothing
# (regularização que evita overconfidence nas predições).
# ═══════════════════════════════════════════════════════════════════════════════
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, smoothing=0.1, num_classes=7):
        super().__init__()
        self.alpha       = alpha
        self.gamma       = gamma
        self.smoothing   = smoothing
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        # Label smoothing: distribuir 'smoothing' uniformemente entre classes
        with torch.no_grad():
            smooth_targets = torch.full_like(inputs, self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        # Cross entropy com targets suavizados
        log_probs = F.log_softmax(inputs, dim=-1)
        ce = -(smooth_targets * log_probs).sum(dim=-1)  # (N,)

        # Fator focal usando targets originais
        probs = F.softmax(inputs, dim=-1)
        pt    = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = self.alpha * (1 - pt) ** self.gamma

        return (focal_weight * ce).mean()

criterion = FocalLossWithSmoothing(
    alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
    smoothing=LABEL_SMOOTHING, num_classes=NUM_CLASSES
)
print(f'Loss: FocalLoss(α={FOCAL_ALPHA}, γ={FOCAL_GAMMA}) + LabelSmoothing({LABEL_SMOOTHING})')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TREINO / INFERÊNCIA — com gradient accumulation
# ═══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, leave=False)):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        if USE_FP16:
            with torch.cuda.amp.autocast():
                logits = model(**kwargs).logits
                loss   = criterion(logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            logits = model(**kwargs).logits
            loss   = criterion(logits, labels) / GRAD_ACCUM_STEPS
            loss.backward()

        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            if USE_FP16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        probs = F.softmax(model(**kwargs).logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
        if 'labels' in batch:
            all_labels.extend(batch['labels'].numpy())
    return np.array(all_probs), np.array(all_labels) if all_labels else None


def compute_f1(probs, labels):
    return f1_score(labels, probs.argmax(axis=1), average='macro')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    return (probs * np.array(thresholds)).argmax(axis=1)

def optimize_temperature(oof_probs, oof_labels):
    def neg_f1(temp):
        return -f1_score(oof_labels, temperature_scale(oof_probs, temp).argmax(axis=1), average='macro')
    return minimize_scalar(neg_f1, bounds=(0.1, 5.0), method='bounded').x

def optimize_thresholds(cal_probs, oof_labels, n_steps=40):
    best_thresholds = [1.0] * NUM_CLASSES
    best_f1 = f1_score(oof_labels, cal_probs.argmax(axis=1), average='macro')
    grid = np.linspace(0.05, 3.0, n_steps)
    for c in range(NUM_CLASSES):
        best_t_c = best_thresholds[c]
        for t in grid:
            thresh_try    = best_thresholds.copy()
            thresh_try[c] = t
            f1 = f1_score(oof_labels, apply_thresholds(cal_probs, thresh_try), average='macro')
            if f1 > best_f1:
                best_f1, best_t_c = f1, t
        best_thresholds[c] = best_t_c
    return best_thresholds, best_f1

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5-FOLD CV TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.save_pretrained(WEIGHTS_DIR / 'tokenizer')
print(f'Tokenizer salvo.')

train_arr = np.array(train_texts)
label_arr = np.array(train_labels)

test_ds     = MammographyDataset(test_texts, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

skf           = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_probs    = np.zeros((len(test_df), NUM_CLASSES))
oof_probs_all = np.zeros((len(train_df), NUM_CLASSES))
all_results   = {}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_arr, label_arr)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*60}')

    X_tr, y_tr = train_arr[train_idx].tolist(), label_arr[train_idx].tolist()
    X_vl, y_vl = train_arr[val_idx].tolist(),   label_arr[val_idx].tolist()

    train_ds = MammographyDataset(X_tr, y_tr, tokenizer, MAX_LEN)
    val_ds   = MammographyDataset(X_vl, y_vl, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

    config = AutoConfig.from_pretrained(MODEL_PATH, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, config=config).to(DEVICE)

    if fold == 0:
        config.save_pretrained(WEIGHTS_DIR / 'model_config')

    # Cosine schedule com warmup (diferente do linear do v5)
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps     = steps_per_epoch * EPOCHS
    warmup_steps    = int(WARMUP_RATIO * total_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler() if USE_FP16 else None

    best_f1, best_state = 0.0, None

    for epoch in range(EPOCHS):
        loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        val_probs, val_lbls = evaluate(model, val_loader)
        val_f1 = compute_f1(val_probs, val_lbls)
        print(f'  Epoch {epoch+1}/{EPOCHS} | loss={loss:.4f} | val_f1={val_f1:.5f}')
        if val_f1 > best_f1:
            best_f1    = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'  Melhor OOF F1 fold {fold}: {best_f1:.5f}')
    torch.save(best_state, WEIGHTS_DIR / f'fold_{fold}.pt')

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    val_probs_best, val_lbls_best = evaluate(model, val_loader)
    oof_probs_all[val_idx] = val_probs_best

    fold_test_probs, _ = evaluate(model, test_loader)
    test_probs += fold_test_probs

    all_results[f'fold_{fold}'] = {
        'val_probs':  val_probs_best.tolist(),
        'val_labels': val_lbls_best.tolist(),
        'f1_macro':   float(best_f1),
        'val_idx':    val_idx.tolist(),
    }

    del model, best_state, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

test_probs /= N_FOLDS
oof_f1 = compute_f1(oof_probs_all, label_arr)
print(f'\n✅ OOF F1-macro (5 folds): {oof_f1:.5f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO + ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════
print('Otimizando temperatura...')
best_temp      = optimize_temperature(oof_probs_all, label_arr)
oof_calibrated = temperature_scale(oof_probs_all, best_temp)
print(f'  Temperatura: {best_temp:.4f} | F1: {compute_f1(oof_calibrated, label_arr):.5f}')

print('Otimizando thresholds...')
best_thresholds, f1_final = optimize_thresholds(oof_calibrated, label_arr)
print(f'  Thresholds: {[round(t, 4) for t in best_thresholds]}')
print(f'  F1 final:   {f1_final:.5f}')

solo_artifacts = {'temperature': best_temp, 'thresholds': best_thresholds, 'oof_f1_macro': oof_f1}
with open(OUTPUT_BASE / 'solo_bert_artifacts.pkl', 'wb') as f:
    pickle.dump(solo_artifacts, f)
with open(OUTPUT_BASE / 'all_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print('Artifacts salvos.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUBMISSION + VERIFICAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════
predictions = apply_thresholds(temperature_scale(test_probs, best_temp), best_thresholds)
submission  = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('=== SUBMISSION ===')
print(submission.to_string(index=False))

print(f'\nEstrutura do output:')
total = 0
for root, dirs, files in os.walk(OUTPUT_BASE):
    dirs.sort()
    indent = '  ' * len(Path(root).relative_to(OUTPUT_BASE).parts)
    print(f'{indent}{Path(root).name}/')
    for fname in sorted(files):
        fpath = Path(root) / fname
        size  = fpath.stat().st_size
        total += size
        print(f'{indent}  {fname}  ({size/1024/1024:.1f} MB)')
print(f'\nTotal: {total/1024/1024:.0f} MB')
print(f'\n✅ Treino v6 concluído!')
print(f'   OOF F1:      {oof_f1:.5f}')
print(f'   Temperatura: {best_temp:.4f}')
print(f'   Thresholds:  {[round(t,4) for t in best_thresholds]}')
print(f'\n➡️  Próximos passos:')
print(f'   1. Save Version → Save & Run All (Commit)')
print(f'   2. Output → New Dataset → nome: model-v{VERSION}')
print(f'   3. Ensemble v5+v6 = 10 folds para máxima diversidade!')